In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms.functional as F
from torch.utils.data import DataLoader, Subset, Dataset
import random
import numpy as np
from PIL import Image

# Control randomness (roughly)
seed = 123
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# Device configuration (GPU if available, otherwise CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Hyperparameters
learning_rate = 0.001
batch_size = 64
num_epochs = 20
image_size = 128  # Resize images to 128x128 for speed
num_classes = 3   # 0: Background, 1: Pet, 2: Border

# ==================== Data Preparation ====================

# Custom Dataset to handle Image and Mask transforms simultaneously
class PetSegmentationDataset(Dataset):
    def __init__(self, root, split='trainval', download=True, size=128):
        # Download Oxford-IIIT Pet dataset
        # split='trainval' for training, 'test' for testing
        self.base_dataset = torchvision.datasets.OxfordIIITPet(root=root,
                                                               split=split,
                                                               target_types='segmentation',
                                                               download=download)
        self.size = size

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        image, mask = self.base_dataset[idx]

        # Resize image and mask
        # Image: Bilinear interpolation (smooth)
        image = F.resize(image, (self.size, self.size), interpolation=F.InterpolationMode.BILINEAR)
        # Mask: Nearest neighbor (discrete values) - Crucial for segmentation labels
        mask = F.resize(mask, (self.size, self.size), interpolation=F.InterpolationMode.NEAREST)

        # Transform to Tensor
        image = F.to_tensor(image) # normalize the range to [0, 1] with float32
        # Normalize image (mean/std roughly for RGB)
        image = F.normalize(image, (0.5, 0.5, 0.5), (0.5, 0.5, 0.5))

        # Mask processing
        # The original dataset has labels: 1 (pet), 2 (background), 3 (border)
        # We convert to 0-indexed: 0 (pet), 1 (background), 2 (border) for CrossEntropyLoss
        mask = torch.as_tensor(np.array(mask), dtype=torch.long) - 1
        # Fix potential bounds if raw data has issues (ensure 0, 1, 2)
        mask = torch.clamp(mask, 0, 2)

        return image, mask

# 1. Load full datasets first
print("Loading datasets...")
train_dataset_full = PetSegmentationDataset(root='./data', split='trainval', download=True, size=image_size)
test_dataset_full = PetSegmentationDataset(root='./data', split='test', download=True, size=image_size)

# 2. Create Subsets
# Control randomness for reproducibility
g = torch.Generator()
g.manual_seed(seed)
train_subset_size = 2500
test_subset_size = 1000
train_indices = torch.randperm(len(train_dataset_full), generator=g)[:train_subset_size]
test_indices = torch.randperm(len(test_dataset_full), generator=g)[:test_subset_size]

train_dataset = Subset(train_dataset_full, train_indices)
test_dataset = Subset(test_dataset_full, test_indices)

# Data loader
train_loader = DataLoader(dataset=train_dataset,
                          batch_size=batch_size,
                          shuffle=True)
test_loader = DataLoader(dataset=test_dataset,
                         batch_size=batch_size,
                         shuffle=False)

# ==================== Model Definition ====================

# A simple U-Net like architecture (Encoder-Decoder)
class SegNet(nn.Module):
    def __init__(self, num_classes):
        super(SegNet, self).__init__()

        # Encoder (Downsampling)
        self.enc1 = self.conv_block(3, 16)
        self.pool1 = nn.MaxPool2d(2) # 128x128 -> 64x64
        self.enc2 = self.conv_block(16, 32)
        self.pool2 = nn.MaxPool2d(2) # 64x64 -> 32x32

        # Bottleneck
        self.bottleneck = self.conv_block(16, 32)

        # Decoder (Upsampling)
        self.upconv1 = nn.ConvTranspose2d(32, 16, kernel_size=2, stride=2) # 64x64 -> 128x128
        self.dec1 = self.conv_block(16, 16)
        self.upconv2 = nn.ConvTranspose2d(16, 16, kernel_size=2, stride=2) #
        self.dec2 = self.conv_block(16, 16)

        # Final classifier (1x1 conv)
        self.final_conv = nn.Conv2d(16, num_classes, kernel_size=1)

    def conv_block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding='same'),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding='same'),
            nn.BatchNorm2d(out_ch),
            nn.ReLU()
        )

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        p1 = self.pool1(e1)

        # Bottleneck
        b = self.bottleneck(p1)

        # Decoder
        d1 = self.upconv1(b)
        d1 = self.dec1(d1)

        out = self.final_conv(d1)
        return out

# Create model instance
model = SegNet(num_classes=num_classes).to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

print(f"Model architecture created. Input size: {image_size}x{image_size}")

# ==================== Training the model ====================

model.train() # Training mode
total_step = len(train_loader)

for epoch in range(num_epochs):
    sum_epoch_loss = 0.0
    for i, (images, masks) in enumerate(train_loader):
        images = images.to(device)
        masks = masks.to(device) # Shape: (Batch, Height, Width)

        # Forward pass
        outputs = model(images) # Shape: (Batch, Num_classes, Height, Width)

        # CrossEntropyLoss
        loss = criterion(outputs, masks)
        sum_epoch_loss += loss.item()

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # Print epoch average loss
    avg_epoch_loss = sum_epoch_loss / total_step
    print(f"Epoch [{epoch+1}/{num_epochs}], Average Loss: {avg_epoch_loss:.4f}")

# ==================== Testing the model ====================

model.eval()  # Testing mode
with torch.no_grad():
    total_iou = 0.0
    num_samples = 0

    for images, masks in test_loader:
        images, masks = images.to(device), masks.to(device)

        # Get predicted label
        outputs = model(images) # Shape: (Batch, Num_classes, Height, Width)
        predicted_labels = outputs.argmax(dim=1) # Shape: (Batch, Height, Width)

        # Calculate IoU for the 'Pet' class (index 1)
        # We convert the multi-class problem to binary (Pet vs. Non-Pet) for IoU calculation
        pred_pet = (predicted_labels == 1).float() # Predicted pet region
        true_pet = (masks == 1).float() # Ground truth pet region

        # Calculate IoU per image in the batch
        for p, t in zip(pred_pet, true_pet):
            intersection = (p * t).sum()
            union = p.sum() + t.sum() - intersection

            # Avoid division by zero
            if union > 0:
                iou = intersection / union
                total_iou += iou.item()
                num_samples += 1
            else:
                # If union is 0 (no pet in both prediction and ground truth),
                # we consider it a perfect match (IoU = 1.0)
                total_iou += 1.0
                num_samples += 1

    # Print average IoU
    avg_iou = total_iou / num_samples
    print(f'Average IoU (Pet Class) on the {len(test_dataset)} test images: {avg_iou * 100:.2f} %')

Using device: cuda
Loading datasets...
Model architecture created. Input size: 128x128
Epoch [1/20], Average Loss: 0.9815
Epoch [2/20], Average Loss: 0.8085
Epoch [3/20], Average Loss: 0.7298
Epoch [4/20], Average Loss: 0.6952
Epoch [5/20], Average Loss: 0.6752
Epoch [6/20], Average Loss: 0.6474
Epoch [7/20], Average Loss: 0.6319
Epoch [8/20], Average Loss: 0.6342
Epoch [9/20], Average Loss: 0.6177
Epoch [10/20], Average Loss: 0.6060
Epoch [11/20], Average Loss: 0.5950
Epoch [12/20], Average Loss: 0.5840
Epoch [13/20], Average Loss: 0.5800
Epoch [14/20], Average Loss: 0.5853
Epoch [15/20], Average Loss: 0.5749
Epoch [16/20], Average Loss: 0.5728
Epoch [17/20], Average Loss: 0.5651
Epoch [18/20], Average Loss: 0.5613
Epoch [19/20], Average Loss: 0.5531
Epoch [20/20], Average Loss: 0.5494
Average IoU (Pet Class) on the 1000 test images: 71.62 %
